# 04 · Ajuste, modelo final y evaluación

Ajustar LightGBM y la regresión logística, elegir el modelo, evaluarlo una sola vez en test y guardarlo.

In [1]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

from src.preprocesamiento import COLUMNAS, crear_preprocesador

optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('../data/processed/train.csv')
y_train = train['FraudFound_P']

versiones = {'con Fault': COLUMNAS, 'sin Fault': [c for c in COLUMNAS if c != 'Fault']}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

def pr_auc(modelo, columnas):
    return cross_validate(modelo, train[columnas], y_train, cv=cv, scoring='average_precision')['test_score'].mean()

## 1. Ajuste de hiperparámetros con Optuna

LightGBM (50 intentos) y regresión logística (20 intentos, solo C), con y sin Fault.

In [2]:
def crear_lgbm(p, columnas):
    return make_pipeline(crear_preprocesador(columnas), LGBMClassifier(**p, subsample_freq=1, random_state=42, verbose=-1, n_jobs=-1))

def crear_logistica(p, columnas):
    return make_pipeline(crear_preprocesador(columnas), SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegression(**p, max_iter=2000))

def espacio_lgbm(t):
    return {'n_estimators': t.suggest_int('n_estimators', 50, 500),
            'learning_rate': t.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': t.suggest_int('num_leaves', 4, 64, log=True),
            'min_child_samples': t.suggest_int('min_child_samples', 10, 200, log=True),
            'subsample': t.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': t.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_lambda': t.suggest_float('reg_lambda', 1e-3, 10, log=True)}

def espacio_logistica(t):
    return {'C': t.suggest_float('C', 1e-3, 10, log=True)}

def ajustar(crear, espacio, columnas, intentos):
    estudio = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(lambda t: pr_auc(crear(espacio(t), columnas), columnas), n_trials=intentos)
    return estudio

candidatos = {'LightGBM': (crear_lgbm, espacio_lgbm, 50), 'Regresión logística': (crear_logistica, espacio_logistica, 20)}

estudios = {}
for version, columnas in versiones.items():
    for nombre, (crear, espacio, intentos) in candidatos.items():
        estudios[(version, nombre)] = ajustar(crear, espacio, columnas, intentos)
        print(version, '|', nombre, '| PR-AUC:', round(estudios[(version, nombre)].best_value, 3))

con Fault | LightGBM | PR-AUC: 0.245


con Fault | Regresión logística | PR-AUC: 0.169


sin Fault | LightGBM | PR-AUC: 0.167


sin Fault | Regresión logística | PR-AUC: 0.146


In [3]:
pd.DataFrame([{'versión': v, 'modelo': m, 'PR-AUC ajustado': e.best_value, **e.best_params}
              for (v, m), e in estudios.items()]).round(3)

,versión,modelo,PR-AUC ajustado,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda,C
0,con Fault,LightGBM,0.245,197.0,0.023,17.0,11.0,0.926,0.470,0.947,NaN
1,con Fault,Regresión logística,0.169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.915
2,sin Fault,LightGBM,0.167,133.0,0.011,12.0,10.0,0.730,0.508,0.721,NaN
3,sin Fault,Regresión logística,0.146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002


## 2. Elección del modelo

LightGBM supera a la regresión logística en las dos versiones (con Fault 0,245 frente a 0,169; sin Fault 0,167 frente a 0,146) y mejora respecto a sus valores por defecto (0,222 y 0,142) con árboles pequeños y aprendizaje lento.

Decisión: LightGBM ajustado como modelo final.

In [4]:
modelos_finales = {v: crear_lgbm(estudios[(v, 'LightGBM')].best_params, columnas) for v, columnas in versiones.items()}

## 3. Umbral de decisión

Calibración: probabilidad media predicha frente a tasa real de fraude por decil.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
prob_oof = {v: cross_val_predict(modelos_finales[v], train[columnas], y_train, cv=cv5, method='predict_proba')[:, 1]
            for v, columnas in versiones.items()}

calibracion = pd.DataFrame({'predicha': prob_oof['con Fault'], 'real': y_train})
calibracion['grupo'] = pd.qcut(calibracion['predicha'], 10, labels=False, duplicates='drop')
calibracion.groupby('grupo')[['predicha', 'real']].mean().round(3)

,predicha,real
grupo,,
0,0.003,0.000
1,0.004,0.000
2,0.006,0.004
3,0.008,0.008
4,0.011,0.008
5,0.037,0.040
6,0.080,0.095
7,0.100,0.118
8,0.128,0.105


Las probabilidades están bien calibradas (21,6 % predicho frente a 22,0 % real en el decil más alto): se pueden usar sin recalibrar.

Umbral por coste: se investiga si la probabilidad supera c / L, con L = 3,75 y varios valores de c.

In [6]:
def evaluar_umbral(y, prob, umbral):
    marcadas = prob >= umbral
    return {'% investigadas': marcadas.mean(),
            'recall': y[marcadas].sum() / y.sum(),
            'precision': y[marcadas].mean() if marcadas.any() else np.nan}

L = 3.75
filas = []
for v in versiones:
    for c in [0.01, 0.03, 0.05, 0.10, 0.20]:
        filas.append({'versión': v, 'c': c, 'umbral c/L': c / L, **evaluar_umbral(y_train, prob_oof[v], c / L)})
pd.DataFrame(filas).round(3)

,versión,c,umbral c/L,% investigadas,recall,precision
0,con Fault,0.01,0.003,0.960,1.000,0.062
1,con Fault,0.03,0.008,0.668,0.991,0.089
2,con Fault,0.05,0.013,0.501,0.967,0.116
3,con Fault,0.10,0.027,0.458,0.958,0.125
4,con Fault,0.20,0.053,0.428,0.930,0.130
5,sin Fault,0.01,0.003,1.000,1.000,0.060
6,sin Fault,0.03,0.008,1.000,1.000,0.060
7,sin Fault,0.05,0.013,1.000,1.000,0.060
8,sin Fault,0.10,0.027,0.686,0.961,0.084
9,sin Fault,0.20,0.053,0.647,0.939,0.087


Con Fault y c = 3 % (umbral 0,8 %) se investiga el 67 % de las reclamaciones y se captura el 99 % del fraude: se puede descartar un tercio perdiendo cerca del 1 %. Sin Fault, con c hasta el 5 % compensa investigarlo todo. Como investigar es barato frente al fraude, el uso más útil del modelo es ordenar las reclamaciones cuando la capacidad es limitada.

Umbral por capacidad: fraude capturado según el porcentaje de reclamaciones revisadas.

In [7]:
def recall_top(y, prob, k):
    top = np.argsort(-prob)[:int(len(y) * k)]
    return np.asarray(y)[top].sum() / np.sum(y)

pd.DataFrame({v: {f'top {int(k*100)} %': recall_top(y_train, prob_oof[v], k) for k in [0.05, 0.10, 0.20, 0.30, 0.50]}
              for v in versiones}).round(3)

,con Fault,sin Fault
top 5 %,0.218,0.172
top 10 %,0.369,0.257
top 20 %,0.545,0.417
top 30 %,0.741,0.542
top 50 %,0.966,0.770


Con Fault, revisando el 20 % de mayor probabilidad se captura el 55 % del fraude, 2,7 veces más que al azar.

## 4. Evaluación final en test

Modelos entrenados con todo train y evaluados una sola vez.

In [8]:
from sklearn.metrics import average_precision_score, roc_auc_score

test = pd.read_csv('../data/processed/test.csv')
y_test = test['FraudFound_P']

resultados, prob_test = {}, {}
for v, columnas in versiones.items():
    modelos_finales[v].fit(train[columnas], y_train)
    prob_test[v] = modelos_finales[v].predict_proba(test[columnas])[:, 1]
    resultados[v] = {'PR-AUC validación': estudios[(v, 'LightGBM')].best_value,
                     'PR-AUC test': average_precision_score(y_test, prob_test[v]),
                     'ROC-AUC test': roc_auc_score(y_test, prob_test[v]),
                     **{f'recall top {int(k*100)} %': recall_top(y_test, prob_test[v], k) for k in [0.05, 0.10, 0.20, 0.30]}}

pd.DataFrame(resultados).round(3)

,con Fault,sin Fault
PR-AUC validación,0.245,0.167
PR-AUC test,0.257,0.150
ROC-AUC test,0.833,0.748
recall top 5 %,0.227,0.162
recall top 10 %,0.378,0.286
recall top 20 %,0.595,0.459
recall top 30 %,0.757,0.616


Con Fault: PR-AUC 0,257 (4,3 veces el azar), ROC-AUC 0,833 y 60 % del fraude en el top 20 %; igual o mejor que en validación (0,245).

Sin Fault: PR-AUC 0,150 (2,5 veces el azar), ROC-AUC 0,748 y 46 % en el top 20 %; algo por debajo de la validación (0,167), lo esperable al elegir el mejor de 50 intentos.

Sin Fault el modelo pierde cerca del 40 % de su capacidad.

In [9]:
pd.DataFrame({v: evaluar_umbral(y_test, prob_test[v], 0.03 / L) for v in versiones}).round(3)

,con Fault,sin Fault
% investigadas,0.704,1.00
recall,1.000,1.00
precision,0.085,0.06


Con el umbral del caso base (c = 3 %), el modelo con Fault investiga el 70 % de las reclamaciones de test y captura todo su fraude; el modelo sin Fault las marca todas.

## 5. Guardado

In [10]:
import joblib

joblib.dump(modelos_finales['con Fault'], '../models/modelo_con_fault.joblib')
joblib.dump(modelos_finales['sin Fault'], '../models/modelo_sin_fault.joblib')

['../models/modelo_sin_fault.joblib']

Modelos guardados en models/ (preprocesador incluido).